In [1]:
import numpy as np
import tensorflow as tf
import tensorflow.keras as keras
import matplotlib.pyplot as plt
import pickle

In [2]:
from tensorflow.compat.v1 import ConfigProto
import os
config=tf.compat.v1.ConfigProto()
config.gpu_options.per_process_gpu_memory_fraction = 0.9
config.gpu_options.allow_growth=True
sess=tf.compat.v1.Session(config=config) 

In [3]:
from load_network import load_Mix
from train import WarmUpCosine, CustomWeightDecaySGD, RSquared
from save_load import load_noise_if_exists, save_layer_outputs_and_labels, load_layer_outputs_and_labels
from weights_bias_reg import load_wb_if_exists
from network import HierarchicalConvEmbedding, MLP, MixerBlock
from evaluate_reg import evalu_stream_main_selected, evalu_select, eval_acc_select_list_single_thresholds, evalu_prepare, compute_stats

In [ ]:
SAVE_PATH = "utkface_12k_split.npz" 
def load_dataset_if_exists(path=SAVE_PATH):
    """
    Load if file exists, otherwise return None.
    """
    if os.path.exists(path):
        data = np.load(path)
        print("✔ Cached data detected, loading directly")
        return (data["x_train"], data["y_train"],
                data["x_test"], data["y_test"])
    return None

In [5]:
x_train, y_train, x_test, y_test = load_dataset_if_exists()

✔ 检测到缓存数据，已直接加载


In [6]:
model=load_Mix()

In [7]:
pred_model = tf.keras.Model(
    inputs=model.get_layer("reg_dense_0").output,
    outputs=model.output
)

In [8]:
l_label = [1,2,3,4,5,6,7,8,9,12]

In [9]:
layer_list = [model.layers[l].name for l in l_label]

In [10]:
NOISE_DIR = "./noise/"
save_layer_outputs_and_labels(model, x_train, y_train, layer_list, save_dir="D:/Data/Mix-8/layer_cache/base")
for i in range(10):
    NOISE_I_DIR = NOISE_DIR + str(i)
    x_gauss, x_salt, x_move, x_occ = load_noise_if_exists(NOISE_I_DIR)
    #x_attack = np.load(o_path, allow_pickle=True)
    save_layer_outputs_and_labels(model, x_gauss, y_train, layer_list, save_dir="D:/Data/Mix-8/layer_cache/gauss/"+str(i))
    save_layer_outputs_and_labels(model, x_salt, y_train, layer_list, save_dir="D:/Data/Mix-8/layer_cache/salt/"+str(i))
    save_layer_outputs_and_labels(model, x_move, y_train, layer_list, save_dir="D:/Data/Mix-8/layer_cache/move/"+str(i))
    save_layer_outputs_and_labels(model, x_occ, y_train, layer_list, save_dir="D:/Data/Mix-8/layer_cache/occ/"+str(i))

[SAVE] Generating layer outputs for: D:/Data/Mix-8/layer_cache/base
[Saved] hierarchical_conv_embedding: outputs (20000, 8192), labels (20000,)
[Saved] mixer_block_0: outputs (20000, 8192), labels (20000,)
[Saved] mixer_block_1: outputs (20000, 8192), labels (20000,)
[Saved] mixer_block_2: outputs (20000, 8192), labels (20000,)
[Saved] mixer_block_3: outputs (20000, 8192), labels (20000,)
[Saved] mixer_block_4: outputs (20000, 8192), labels (20000,)
[Saved] mixer_block_5: outputs (20000, 8192), labels (20000,)
[Saved] mixer_block_6: outputs (20000, 8192), labels (20000,)
[Saved] mixer_block_7: outputs (20000, 8192), labels (20000,)
[Saved] reg_dense_0: outputs (20000, 128), labels (20000,)
[SAVE] Generating layer outputs for: D:/Data/Mix-8/layer_cache/gauss/0
[Saved] hierarchical_conv_embedding: outputs (20000, 8192), labels (20000,)
[Saved] mixer_block_0: outputs (20000, 8192), labels (20000,)
[Saved] mixer_block_1: outputs (20000, 8192), labels (20000,)
[Saved] mixer_block_2: outputs

In [11]:
for i in range(5):
    path = "./attack/" + str(i)
    ATTACK_DIR = os.path.join(path, "Mix_pgd.npy")
    x_attack = np.load(ATTACK_DIR)
    save_layer_outputs_and_labels(model, x_attack, y_train, layer_list, save_dir="D:/Data/Mix-8/layer_cache/attack/"+str(i))

[SAVE] Generating layer outputs for: D:/Data/Mix-8/layer_cache/attack/0
[Saved] hierarchical_conv_embedding: outputs (20000, 8192), labels (20000,)
[Saved] mixer_block_0: outputs (20000, 8192), labels (20000,)
[Saved] mixer_block_1: outputs (20000, 8192), labels (20000,)
[Saved] mixer_block_2: outputs (20000, 8192), labels (20000,)
[Saved] mixer_block_3: outputs (20000, 8192), labels (20000,)
[Saved] mixer_block_4: outputs (20000, 8192), labels (20000,)
[Saved] mixer_block_5: outputs (20000, 8192), labels (20000,)
[Saved] mixer_block_6: outputs (20000, 8192), labels (20000,)
[Saved] mixer_block_7: outputs (20000, 8192), labels (20000,)
[Saved] reg_dense_0: outputs (20000, 128), labels (20000,)
[SAVE] Generating layer outputs for: D:/Data/Mix-8/layer_cache/attack/1
[Saved] hierarchical_conv_embedding: outputs (20000, 8192), labels (20000,)
[Saved] mixer_block_0: outputs (20000, 8192), labels (20000,)
[Saved] mixer_block_1: outputs (20000, 8192), labels (20000,)
[Saved] mixer_block_2: ou

In [12]:
CACHE_DIR = "./Mix-8/w_and_b_cache"

In [13]:
eva_w,eva_b = load_wb_if_exists(y_train, layer_list, CACHE_DIR, save_dir="D:/Data/Mix-8/layer_cache/base")

In [14]:
threshold_list, Y_list = evalu_prepare(y_train, n=9)

In [15]:
select_list = evalu_select(layer_list, y_train, eva_w, eva_b, pred_model=pred_model, save_dir="D:/Data/Mix-8/layer_cache/base")

In [16]:
base = evalu_stream_main_selected(layer_list, y_train, eva_w, eva_b, select_list, save_dir="D:/Data/Mix-8/layer_cache/base")
base_final = eval_acc_select_list_single_thresholds(model, x_train, 'train', select_list, threshold_list) 
base = np.concatenate((base,base_final.reshape(1,9)),axis=0)

Layer 0
accuracy: [0.84387831 0.71755976 0.63425722 0.62045524 0.64266758 0.56233647
 0.59478839 0.67560682 0.80144377]
Layer 1
accuracy: [0.93511674 0.84473814 0.73160479 0.68895513 0.7100944  0.61602689
 0.57429011 0.66106995 0.80387071]
Layer 2
accuracy: [0.94497483 0.86808114 0.75484062 0.69846803 0.72979169 0.64590135
 0.58754291 0.61819131 0.77890938]
Layer 3
accuracy: [0.95534988 0.89022521 0.76316871 0.70805717 0.72952645 0.64373016
 0.60003196 0.63454287 0.75155261]
Layer 4
accuracy: [0.9518089  0.90235211 0.77352108 0.71009163 0.72931153 0.62046478
 0.61148598 0.58137663 0.75505779]
Layer 5
accuracy: [0.95452288 0.90668388 0.80040552 0.73201702 0.73913169 0.64190159
 0.61623788 0.63280011 0.77165006]
Layer 6
accuracy: [0.95408518 0.93133761 0.80786877 0.76283172 0.76294567 0.69542167
 0.72393211 0.76738238 0.83454441]
Layer 7
accuracy: [0.94968025 0.94385425 0.87565591 0.79096661 0.77351175 0.70042521
 0.77281339 0.85191109 0.8883204 ]
Layer 8
accuracy: [0.93676947 0.94099594

In [17]:
compute_stats(base)

(array([[0.73189843, 0.60848643, 0.69061299],
        [0.83715322, 0.67169214, 0.67974359],
        [0.85596553, 0.69138702, 0.66154787],
        [0.86958127, 0.69377126, 0.66204248],
        [0.87589403, 0.68662265, 0.6493068 ],
        [0.88720409, 0.7043501 , 0.67356268],
        [0.89776386, 0.74039969, 0.7752863 ],
        [0.92306347, 0.75496785, 0.83768162],
        [0.93051372, 0.75261128, 0.85660546],
        [0.96720517, 0.73182598, 0.85751071],
        [1.        , 1.        , 1.        ]]),
 array([[0.08617598, 0.03386947, 0.08503137],
        [0.08325634, 0.04029625, 0.09465144],
        [0.0780933 , 0.03461217, 0.08392506],
        [0.07980405, 0.03645377, 0.06484237],
        [0.07515167, 0.04743416, 0.07578081],
        [0.06440826, 0.04425319, 0.06968705],
        [0.06424021, 0.0318043 , 0.04550182],
        [0.03360648, 0.03922026, 0.04821705],
        [0.01196065, 0.0576496 , 0.03854132],
        [0.01149394, 0.04954695, 0.03149358],
        [0.        , 0.        ,

In [18]:
attack = np.zeros((len(layer_list),9))
attack_final = np.zeros(9)
for i in range(5):
    attack += evalu_stream_main_selected(layer_list, y_train, eva_w, eva_b, select_list, save_dir = "D:/Data/Mix-8/layer_cache/attack/"+str(i))
attack = attack/5
for i in range(5):
    path = "./attack/" + str(i)
    ATTACK_DIR = os.path.join(path, "Mix_pgd.npy")
    attack_final += eval_acc_select_list_single_thresholds(model, ATTACK_DIR, 'train', select_list, threshold_list)
attack_final = attack_final/5
attack = np.concatenate((attack,attack_final.reshape(1,9)),axis=0)

Layer 0
accuracy: [0.82163856 0.69104843 0.61319569 0.60323475 0.60752806 0.44036455
 0.4863449  0.76879096 0.88165652]
Layer 1
accuracy: [0.92656966 0.808272   0.72163331 0.61974148 0.5242528  0.54240784
 0.64305462 0.78318923 0.89150185]
Layer 2
accuracy: [0.92971602 0.80705697 0.7220781  0.61675578 0.49743733 0.53453914
 0.64369806 0.78149309 0.90334061]
Layer 3
accuracy: [0.93013797 0.80561699 0.71619534 0.61708435 0.48654692 0.52118088
 0.6669923  0.80562041 0.90975011]
Layer 4
accuracy: [0.93013797 0.80971404 0.71597208 0.6036483  0.47905653 0.45030516
 0.65388997 0.78370528 0.90995055]
Layer 5
accuracy: [0.93013797 0.80622501 0.72127644 0.61875991 0.47493211 0.40191612
 0.59361995 0.77522275 0.90659853]
Layer 6
accuracy: [0.93013797 0.81074746 0.70518058 0.59930094 0.45733994 0.35576388
 0.54702132 0.76765988 0.90372031]
Layer 7
accuracy: [0.93013797 0.8072578  0.69057702 0.58629501 0.44629756 0.31029802
 0.35312204 0.63240777 0.87560864]
Layer 8
accuracy: [0.93013797 0.81218595

In [19]:
compute_stats(attack)

(array([[0.70861445, 0.55031696, 0.71219987],
        [0.81868389, 0.56233887, 0.77248196],
        [0.81970009, 0.54948102, 0.77624836],
        [0.81731617, 0.541616  , 0.79403114],
        [0.81853796, 0.51079953, 0.78206937],
        [0.81925009, 0.49824355, 0.75860604],
        [0.81531354, 0.47086825, 0.73897245],
        [0.8093853 , 0.44775318, 0.61983433],
        [0.80566276, 0.44182534, 0.55352377],
        [0.66450104, 0.41436177, 0.19327802],
        [0.66539967, 0.41553031, 0.19470449]]),
 array([[0.08609041, 0.07789817, 0.16619833],
        [0.08412634, 0.04139968, 0.10185981],
        [0.08526011, 0.04993531, 0.10592385],
        [0.08773333, 0.05523057, 0.09973465],
        [0.08774081, 0.06651048, 0.10487665],
        [0.08577239, 0.09035684, 0.12829652],
        [0.09196257, 0.09994726, 0.14720787],
        [0.09763915, 0.11279382, 0.2136766 ],
        [0.1043619 , 0.11044004, 0.18828648],
        [0.07216577, 0.08368759, 0.03078348],
        [0.07122128, 0.08467268,

In [20]:
gauss = np.zeros((len(layer_list),9))
gauss_final = np.zeros(9)
for i in range(10):
    gauss += evalu_stream_main_selected(layer_list, y_train, eva_w, eva_b, select_list, save_dir = "D:/Data/Mix-8/layer_cache/gauss/"+str(i))
gauss = gauss/10
for i in range(10):
    path = "./noise/" + str(i)
    GAUSS_DIR = os.path.join(path, "gauss.npy")
    gauss_final += eval_acc_select_list_single_thresholds(model, GAUSS_DIR, 'train', select_list, threshold_list)
gauss_final = gauss_final/10
gauss = np.concatenate((gauss,gauss_final.reshape(1,9)),axis=0)

Layer 0
accuracy: [0.34852681 0.28989777 0.29068779 0.36575686 0.47444668 0.38239522
 0.28376892 0.82040245 0.91384213]
Layer 1
accuracy: [0.9299354  0.81464399 0.73708942 0.64568188 0.62917014 0.62114092
 0.71245805 0.81839034 0.91154681]
Layer 2
accuracy: [0.93013797 0.81547229 0.73687145 0.64307594 0.59061969 0.61983997
 0.71515915 0.82020903 0.91363771]
Layer 3
accuracy: [0.93013797 0.81526804 0.73667774 0.64291456 0.56820698 0.61913796
 0.71578624 0.82040245 0.91384213]
Layer 4
accuracy: [0.93013797 0.81526804 0.73667774 0.64316734 0.56310863 0.62123934
 0.71557721 0.82040245 0.91384213]
Layer 5
accuracy: [0.93013797 0.81526804 0.73667774 0.64291456 0.55885305 0.61512264
 0.71492095 0.82040245 0.91384213]
Layer 6
accuracy: [0.93013797 0.81526804 0.73667774 0.64291456 0.55579427 0.53287001
 0.70470975 0.82065237 0.91384213]
Layer 7
accuracy: [0.93013797 0.81526804 0.73667774 0.64332258 0.54842356 0.4141617
 0.50560517 0.76438842 0.90704086]
Layer 8
accuracy: [0.93013797 0.81526804 

In [21]:
compute_stats(gauss)

(array([[0.30911254, 0.40769364, 0.67285766],
        [0.82744197, 0.63237702, 0.81435844],
        [0.82761021, 0.61838781, 0.81638608],
        [0.82736125, 0.61063116, 0.81665589],
        [0.82736125, 0.60999662, 0.81666971],
        [0.82736125, 0.60459681, 0.81670924],
        [0.82736852, 0.57611138, 0.8137164 ],
        [0.82736852, 0.53471578, 0.72779286],
        [0.82740343, 0.53068065, 0.66595456],
        [0.82927179, 0.53274419, 0.30439916],
        [0.82858725, 0.53017269, 0.28101458]]),
 array([[0.02647793, 0.04707393, 0.27736736],
        [0.07889342, 0.01030735, 0.08155283],
        [0.07925158, 0.02126413, 0.08113853],
        [0.07944138, 0.03096986, 0.08092513],
        [0.07944138, 0.03343738, 0.0809079 ],
        [0.07944138, 0.03496302, 0.0808586 ],
        [0.07943308, 0.04839879, 0.08451128],
        [0.07943308, 0.09508626, 0.16329803],
        [0.07939324, 0.1003516 , 0.17523168],
        [0.07850846, 0.10378788, 0.02593414],
        [0.07862021, 0.10427532,

In [22]:
salt = np.zeros((len(layer_list),9))
salt_final = np.zeros(9)
for i in range(10):
    salt += evalu_stream_main_selected(layer_list, y_train, eva_w, eva_b, select_list, save_dir = "D:/Data/Mix-8/layer_cache/salt/"+str(i))
salt = salt/10
for i in range(10):
    path = "./noise/" + str(i)
    SALT_DIR = os.path.join(path, "salt.npy")
    salt_final += eval_acc_select_list_single_thresholds(model, SALT_DIR, 'train', select_list, threshold_list)
salt_final = salt_final/10
salt = np.concatenate((salt,salt_final.reshape(1,9)),axis=0)

Layer 0
accuracy: [0.1119352  0.19170677 0.26521931 0.35768757 0.45885215 0.38217398
 0.28421376 0.82040245 0.91384213]
Layer 1
accuracy: [0.93012435 0.81526473 0.73453199 0.66304137 0.65499471 0.61226527
 0.65533322 0.77353411 0.87408991]
Layer 2
accuracy: [0.93013797 0.81670259 0.73771167 0.64326938 0.61328069 0.63181533
 0.71170396 0.81878602 0.91323859]
Layer 3
accuracy: [0.93013797 0.81526804 0.73730419 0.64282565 0.58128477 0.62741071
 0.71514456 0.82020903 0.91384213]
Layer 4
accuracy: [0.93013797 0.81526804 0.73688367 0.6446503  0.57530284 0.63027074
 0.7157894  0.82044388 0.91384213]
Layer 5
accuracy: [0.93013797 0.81526804 0.73667774 0.6433701  0.56900663 0.62910074
 0.71498385 0.82130527 0.91384213]
Layer 6
accuracy: [0.93013797 0.81526804 0.73775426 0.64469783 0.57627852 0.591514
 0.71844291 0.82257371 0.91406695]
Layer 7
accuracy: [0.93013797 0.81526804 0.73838563 0.64428367 0.57007394 0.47336703
 0.62726951 0.80688765 0.90997295]
Layer 8
accuracy: [0.93013797 0.81547586 0

In [23]:
compute_stats(salt)

(array([[0.18804529, 0.39953347, 0.67281945],
        [0.8272056 , 0.64088775, 0.76820321],
        [0.82822862, 0.6284059 , 0.81467417],
        [0.82745958, 0.61662442, 0.81662689],
        [0.82740361, 0.61568216, 0.81668229],
        [0.82739631, 0.61283641, 0.8166673 ],
        [0.82750935, 0.59980521, 0.81819757],
        [0.82780355, 0.56118515, 0.78308281],
        [0.82888048, 0.55267952, 0.75045252],
        [0.83730367, 0.5670881 , 0.52364819],
        [0.8395704 , 0.56929246, 0.50307312]]),
 array([[0.06293842, 0.04327215, 0.27742089],
        [0.07999219, 0.02141713, 0.089957  ],
        [0.07894286, 0.01312492, 0.08234661],
        [0.07935249, 0.02643105, 0.08099171],
        [0.07939988, 0.03059605, 0.08089633],
        [0.07942845, 0.03309175, 0.0810079 ],
        [0.07928599, 0.03135666, 0.07965616],
        [0.07897758, 0.07080181, 0.11719358],
        [0.07784825, 0.08110393, 0.133792  ],
        [0.07659887, 0.07062947, 0.05696124],
        [0.07345514, 0.07628127,

In [24]:
move = np.zeros((len(layer_list),9))
move_final = np.zeros(9)
for i in range(10):
    move += evalu_stream_main_selected(layer_list, y_train, eva_w, eva_b, select_list, save_dir = "D:/Data/Mix-8/layer_cache/move/"+str(i))
move = move/10
for i in range(10):
    path = "./noise/" + str(i)
    MOVE_DIR = os.path.join(path, "move.npy")
    move_final += eval_acc_select_list_single_thresholds(model, MOVE_DIR, 'train', select_list, threshold_list)
move_final = move_final/10
move = np.concatenate((move,move_final.reshape(1,9)),axis=0)

Layer 0
accuracy: [0.92637372 0.81280149 0.73580661 0.64238406 0.56900998 0.61705118
 0.7143402  0.20896039 0.16176084]
Layer 1
accuracy: [0.93055832 0.81750982 0.73748759 0.64356307 0.63133503 0.63512946
 0.70957541 0.81917286 0.91383242]
Layer 2
accuracy: [0.93055673 0.81588381 0.73622839 0.642698   0.63895201 0.63393563
 0.7153415  0.82022896 0.91403683]
Layer 3
accuracy: [0.9309708  0.8171038  0.73516373 0.64888647 0.66095497 0.6289882
 0.7105131  0.8197732  0.91384213]
Layer 4
accuracy: [0.9309708  0.8185378  0.73414859 0.65305699 0.67293277 0.6323012
 0.70915486 0.81761994 0.91384213]
Layer 5
accuracy: [0.93055204 0.81896173 0.73433738 0.65242395 0.6736823  0.64433586
 0.71093504 0.81677339 0.9136397 ]
Layer 6
accuracy: [0.93013797 0.82141759 0.72518959 0.66442077 0.677765   0.67312135
 0.73226481 0.82536692 0.91404654]
Layer 7
accuracy: [0.93034912 0.8228433  0.7377057  0.67077753 0.65975235 0.66923975
 0.74143989 0.8312677  0.91222149]
Layer 8
accuracy: [0.93013797 0.82222257 0

In [25]:
compute_stats(move)

(array([[0.82495093, 0.60951537, 0.36170809],
        [0.82786213, 0.63397259, 0.81487221],
        [0.82767539, 0.64007629, 0.81679931],
        [0.8279769 , 0.64573786, 0.81573246],
        [0.82776159, 0.65330357, 0.81513821],
        [0.8278397 , 0.65625027, 0.81416364],
        [0.82499926, 0.67115986, 0.823235  ],
        [0.82917585, 0.6620935 , 0.82900934],
        [0.83189931, 0.65425128, 0.8259692 ],
        [0.83721934, 0.61925252, 0.77547975],
        [0.83575727, 0.65974899, 0.80789891]]),
 array([[0.07847242, 0.03126657, 0.25002673],
        [0.07947105, 0.00659647, 0.082742  ],
        [0.07968217, 0.00543357, 0.08077243],
        [0.07988643, 0.01223135, 0.08155256],
        [0.08052703, 0.01600585, 0.08140538],
        [0.08036722, 0.0113957 , 0.08181359],
        [0.08443315, 0.00460649, 0.07446114],
        [0.08026658, 0.00786155, 0.07000208],
        [0.0763307 , 0.01745159, 0.07097114],
        [0.07487706, 0.030733  , 0.071896  ],
        [0.08006673, 0.00698477,

In [26]:
occ = np.zeros((len(layer_list),9))
occ_final = np.zeros(9)
for i in range(10):
    occ += evalu_stream_main_selected(layer_list, y_train, eva_w, eva_b, select_list, save_dir = "D:/Data/Mix-8/layer_cache/occ/"+str(i))
occ = occ/10
for i in range(10):
    path = "./noise/" + str(i)
    OCC_DIR = os.path.join(path, "occ.npy")
    occ_final += eval_acc_select_list_single_thresholds(model, OCC_DIR, 'train', select_list, threshold_list)
occ_final = occ_final/10
occ = np.concatenate((occ,occ_final.reshape(1,9)),axis=0)

Layer 0
accuracy: [0.73660291 0.55876241 0.44667182 0.43156125 0.47169145 0.55377746
 0.522616   0.79466982 0.90923328]
Layer 1
accuracy: [0.93258436 0.81958742 0.73972091 0.65034598 0.60730119 0.62407154
 0.70875445 0.81437774 0.91024056]
Layer 2
accuracy: [0.93177267 0.82326978 0.74314774 0.6465684  0.61507084 0.6220433
 0.71104215 0.81830876 0.91343727]
Layer 3
accuracy: [0.93240576 0.82288019 0.74524134 0.64153584 0.61221446 0.62212031
 0.71447263 0.82057337 0.91341488]
Layer 4
accuracy: [0.93240576 0.82266354 0.7479952  0.64732761 0.61132672 0.62135451
 0.71637601 0.82063473 0.91384213]
Layer 5
accuracy: [0.93136842 0.82533475 0.75179878 0.65485722 0.61224169 0.62945409
 0.71700746 0.82127649 0.91384213]
Layer 6
accuracy: [0.93136842 0.82718081 0.74144699 0.64671271 0.59103111 0.6381568
 0.71765578 0.82104792 0.91384213]
Layer 7
accuracy: [0.93219302 0.83433361 0.7469439  0.63979395 0.56799106 0.64934581
 0.72483744 0.8237895  0.91406893]
Layer 8
accuracy: [0.9321817  0.83945168 0

In [27]:
compute_stats(occ)

(array([[0.57570174, 0.48594737, 0.74388288],
        [0.83001452, 0.62705983, 0.81094763],
        [0.83195624, 0.62905099, 0.81507957],
        [0.8316366 , 0.62730403, 0.81648976],
        [0.83211966, 0.62845041, 0.8169386 ],
        [0.83405861, 0.6325783 , 0.81747492],
        [0.83328784, 0.6305748 , 0.81795199],
        [0.83758822, 0.62686885, 0.82165593],
        [0.83398806, 0.61804304, 0.82256848],
        [0.79619384, 0.56458602, 0.79727393],
        [0.79505756, 0.59301908, 0.82264567]]),
 array([[0.12306148, 0.05436366, 0.16000726],
        [0.0796831 , 0.01724651, 0.08280395],
        [0.07815088, 0.01347561, 0.08180914],
        [0.07821724, 0.01223809, 0.08095217],
        [0.07784598, 0.01463386, 0.08062408],
        [0.0758762 , 0.01728598, 0.08025755],
        [0.07753269, 0.02588031, 0.0797218 ],
        [0.07570945, 0.03940327, 0.07625771],
        [0.08067736, 0.04051736, 0.07564975],
        [0.12744169, 0.04139321, 0.07850127],
        [0.12546295, 0.03958399,

In [29]:
SAVE_FILE='Mix-8.pkl'

In [30]:
progress = {"base": base, "attack": attack,"gauss": gauss,"salt": salt,"move": move,"occ": occ}
with open(SAVE_FILE, "wb") as f:
    pickle.dump(progress, f)

In [ ]:
def stats_minmax_from_matrix_dict(matrix_dict, check_num=6): 
    """ 
    Input: 
        matrix_dict: {name: np.ndarray(m, n)}, a dictionary containing 6 matrices 
    Output: 
        stats: { 
            name: { 
                "mean": (m,), 
                "min":  (m,), 
                "max":  (m,) 
            }, ... 
        } 
    For each component i, statistics are computed along axis=1 (over n samples): 
        mean[i] = mean(X[i, :]) 
        min[i]  = min(X[i, :]) 
        max[i]  = max(X[i, :]) 
    """ 
    if check_num is not None and len(matrix_dict) != check_num: 
        raise ValueError(f"Expected {check_num} matrices, but received {len(matrix_dict)}") 
 
    # shape consistency 
    shapes = [np.asarray(v).shape for v in matrix_dict.values()] 
    if len(set(shapes)) != 1: 
        raise ValueError(f"Inconsistent matrix shapes: {shapes}") 
 
    stats = {} 
    for name, X in matrix_dict.items(): 
        X = np.asarray(X) 
        stats[name] = { 
            "mean": X.mean(axis=1), 
            "std":  X.std(axis=1), 
            "min":  X.min(axis=1), 
            "max":  X.max(axis=1), 
        } 
    return stats 

In [33]:
SAVE_FILE='Mix-8.pkl'
with open(SAVE_FILE, "rb") as f:
    progress = pickle.load(f)

In [34]:
mean_var = stats_minmax_from_matrix_dict(progress)

In [35]:
mean_var

{'base': {'mean': array([0.67699928, 0.72952965, 0.73630014, 0.74179834, 0.73727449,
         0.75503896, 0.80448328, 0.83857098, 0.84657682, 0.85218062,
         1.        ]),
  'std': array([0.08887638, 0.10789032, 0.10994677, 0.11090262, 0.11994416,
         0.11200955, 0.08340498, 0.07983684, 0.08352139, 0.10218146,
         0.        ]),
  'min': array([0.56233647, 0.57429011, 0.58754291, 0.60003196, 0.58137663,
         0.61623788, 0.69542167, 0.70042521, 0.67312138, 0.69414476,
         1.        ]),
  'max': array([0.84387831, 0.93511674, 0.94497483, 0.95534988, 0.9518089 ,
         0.95452288, 0.95408518, 0.94968025, 0.94099594, 0.98235557,
         1.        ])},
 'attack': {'mean': array([0.65704376, 0.71783491, 0.71514315, 0.71765444, 0.70380228,
         0.69203323, 0.67505141, 0.6256576 , 0.60033729, 0.42404694,
         0.42521149]),
  'std': array([0.13927659, 0.13723754, 0.14502026, 0.14994437, 0.16295325,
         0.17335106, 0.18758718, 0.21083429, 0.20657226, 0.2035